# Convert a PIVview time series

A plane contains multiple results at one z position. This notebook uses bundled
inputs from a source checkout. The start date and 100 Hz rate are illustrative;
for an experiment, use its actual result timestamps or sampling rate. This rate
is distinct from the pulse separation within an image pair.

In [ ]:
from datetime import datetime
from pathlib import Path
import h5py
from piv2hdf import PIVPlane, set_config, tutorial
from piv2hdf.pivview import PIVViewNcFile
from piv2hdf.pivview.user_operations import add_standard_name_operation

plane = PIVPlane.from_folder(
    tutorial.PIVview.get_plane_directory(),
    time_info=(datetime(2026, 1, 1, 12), 100.0),
    pivfile=PIVViewNcFile,
    parameter=tutorial.PIVview.get_parameter_file(),
    udo=add_standard_name_operation,
)

## Write and validate

The hook labels velocities and relative time, enabling automatic sample means.
The source z position and units are kept. Use a quantity such as `z="5 mm"` on
`to_hdf` only when it represents the actual measurement position.

In [ ]:
with set_config(convention="planar_piv"):
    output = plane.to_hdf(
        hdf_filename=Path("outputs/pivview_plane.hdf"),
        piv_attributes={"creator": {"name": "Example researcher"}},
    )

## Inspect the time axis and means

Velocity shape is `(reltime, y, x)`; means and counts have shape `(y, x)`.
Counts exclude nonfinite and flag-invalid observations. A sample mean does not
weight by elapsed time. For irregular sampling, choose the estimator appropriate
to the experiment; see [statistics](../docs/usage.md#control-validity-and-statistics).

In [ ]:
with h5py.File(output, "r") as data:
    print("velocity shape:", data["u"].shape)
    print("relative times (s):", data["reltime"][...])
    print("mean/count shape:", data["mean_u"].shape, data["mean_u_valid_count"].shape)
    print("estimator:", data["mean_u"].attrs["averaging_method"])